# Isolation Forest con EquiFrame

Simularemos lotes de cien filas. El detector aprende la tasa de nulos de una columna.
Las etiquetas de prueba indican si hubo un fallo de procesamiento y se fijan antes
de entrenar. Un lote válido puede tener muchos nulos, y un fallo puede dejar intacta
la tasa de nulos.

In [1]:
%pip -q install "equiframe[engines,ml]==0.14.0"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
from hashlib import sha256
from importlib.metadata import version

import pandas as pd

import equiframe as ef

assert ef.__version__ == "0.14.0"
print({name: version(name) for name in ("equiframe", "pandas", "scikit-learn", "numpy")})

{'equiframe': '0.14.0', 'pandas': '3.1.1', 'scikit-learn': '1.9.1', 'numpy': '2.5.5'}


## Lotes de entrenamiento, calibración y prueba

Cada lote tiene su propio identificador. Entrenamiento y calibración contienen
solamente lotes normales. La prueba incluye un aumento legítimo de nulos,
otro aumento provocado por un fallo y un fallo que conserva la tasa usual.

In [3]:
spec = ef.ProfileSpec(
    family="nulls_types",
    features=(
        ef.ProfileFeature(
            name="null_rate", metric="tasa_nulos", table="result", columns=("amount",)
        ),
    ),
)
spec_sha256 = ef.profile_spec_fingerprint(spec)


def profile(nulls):
    amount = pd.Series([None] * nulls + [1] * (100 - nulls), dtype="Int64")
    missing, total = int(amount.isna().sum()), len(amount)
    return ef.ProfileVector(
        spec=spec,
        spec_sha256=spec_sha256,
        observations=(
            ef.FeatureObservation(
                name="null_rate",
                numerator=missing,
                denominator=total,
                state="available",
                value=missing / total,
            ),
        ),
    )


def selection_hash(batches):
    ids = [batch_id for batch_id, _ in batches]
    return sha256(json.dumps(ids).encode()).hexdigest()


training = [(f"train-{i}", n) for i, n in enumerate(range(15, 56))]
calibration = [(f"cal-{i}", n) for i, n in enumerate(range(19, 51))]
test = [
    ("normal habitual", 30, False),
    ("aumento legítimo", 95, False),
    ("fallo que introduce nulos", 95, True),
    ("fallo que conserva la tasa", 30, True),
]

## Ajustar el bosque y calibrar el umbral

El umbral se calcula sobre las puntuaciones de calibración con alfa 0,10.
Una puntuación estrictamente mayor que el umbral produce una alerta.

In [4]:
fit = ef.fit_detectors(
    ef.NormalTrainingProfiles(
        purpose="normal_training",
        selection_manifest_sha256=selection_hash(training),
        profiles=tuple(profile(n) for _, n in training),
    ),
    spec=spec,
    config=ef.IsolationForestConfig(random_state=7, n_estimators=64, max_samples=32),
    support=ef.DetectorSupport(
        min_training_rows=32,
        min_calibration_rows=20,
        rationale="Mínimos técnicos del ejemplo sintético",
    ),
)
bundle = ef.calibrate_detectors(
    fit,
    ef.NormalCalibrationProfiles(
        purpose="normal_calibration",
        selection_manifest_sha256=selection_hash(calibration),
        profiles=tuple(profile(n) for _, n in calibration),
    ),
    alpha="0.10",
)
assert fit.m.state == "fitted" and bundle.m.state == "ready"
print(f"Árboles: {len(fit.m.trees)}. Rango de calibración: {bundle.m.rank}/{len(calibration)}.")
print(f"Umbral: {bundle.m.threshold:.6f}")

Árboles: 64. Rango de calibración: 29/32.
Umbral: 0.515061


## Detectar y contrastar con los fallos conocidos

Las etiquetas entran aquí para calcular los aciertos y errores. Contamos
los casos no evaluables por separado. Una falsa alarma es una alerta sobre
un lote normal. Una omisión es un fallo evaluado que quedó sin alerta.

In [5]:
result = ef.detect_component(
    bundle, tuple(profile(n) for _, n, _ in test), method="isolation_forest"
)
rows = pd.DataFrame(
    [
        {
            "lote": label,
            "nulos": n,
            "fallo": fault,
            "puntuación": event.score,
            "decisión": event.result.decision,
        }
        for (label, n, fault), event in zip(test, result.decisions, strict=True)
    ]
)
display(rows)
alert = rows["decisión"].eq("alert")
quiet = rows["decisión"].eq("no_alert")
fault = rows["fallo"]
counts = {
    "fallos detectados": int((alert & fault).sum()),
    "falsas alarmas": int((alert & ~fault).sum()),
    "omisiones": int((quiet & fault).sum()),
    "normales sin alerta": int((quiet & ~fault).sum()),
    "no evaluables": int((~(alert | quiet)).sum()),
}
print(counts)
assert sum(counts.values()) == len(test)

,lote,nulos,fallo,puntuación,decisión
0,normal habitual,30,False,0.460279,no_alert
1,aumento legítimo,95,False,0.663365,alert
2,fallo que introduce nulos,95,True,0.663365,alert
3,fallo que conserva la tasa,30,True,0.460279,no_alert


{'fallos detectados': 1, 'falsas alarmas': 1, 'omisiones': 1, 'normales sin alerta': 1, 'no evaluables': 0}


El aumento legítimo y el fallo que introduce nulos generan el mismo perfil.
Lo mismo ocurre con el lote normal habitual y el fallo que conserva la tasa.
El ejemplo muestra cómo interpretar la puntuación y cómo medir errores en
estos cuatro casos sintéticos. Usa una comparación exacta cuando necesites
comprobar que una transformación conservó los datos.